# Instacart Data Pipeline
## Stage 1: Bronze (Ingestion + Validation)

**Owner:** Nadine  
**Source:** Six CSV files in a Unity Catalog volume  
**Target schema:** `workspace.instacart_bronze`

### What this notebook does

Prepares the pipeline schemas, loads all 6 Bronze tables (`aisles`,
`departments`, `products`, `orders`, `order_products_prior`, and
`order_products_train`), then runs one consolidated validation query
before handing off to Silver.

Bronze is a typed source copy, not cleaned data. These queries do not
trim stored names, remove duplicates, filter invalid records, or join
business tables. They preserve the source records and check for problems.

### Build Order

| # | Task | Depends on | Why |
|---|---|---|---|
| 01 | `setup` | Source CSVs available | Prepares schemas and verifies the six expected filenames |
| 02 | `bronze_aisles` | `setup` | Reads only `aisles.csv` |
| 03 | `bronze_departments` | `setup` | Reads only `departments.csv` |
| 04 | `bronze_products` | `setup` | Reads only `products.csv`; handles embedded quotes |
| 05 | `bronze_orders` | `setup` | Reads only `orders.csv` |
| 06 | `bronze_order_products_prior` | `setup` | Reads only `order_products__prior.csv` |
| 07 | `bronze_order_products_train` | `setup` | Reads only `order_products__train.csv` |
| 08 | `bronze_validation` | All 6 ingestion tasks | Checks the complete Bronze layer and stops on failure |

**Job DAG:** All six ingestion tasks are independent after Setup and can
run in parallel. Validation must depend on all six. Unlike the Silver
builds, no Bronze load reads another Bronze table. Run top-to-bottom when
using this notebook manually.

### Design for scale

Every load follows the same pattern: documented header, target catalog
and schema, then `CREATE OR REPLACE TABLE ... AS SELECT` with an explicit
CSV schema. Repeating a load replaces its contents instead of appending
another copy. Validation uses matching metrics joined with `UNION ALL`.

This is a full-refresh design for the homework snapshot, not incremental
ingestion. Full reloads and exact duplicate checks become more expensive
as data grows; more frequent production arrivals would need a different
loading strategy.

### Why `_rescued_data` is retained

The reader generates this field; it is not in the original CSV.
It retains information rescued when source content does not fit the
declared schema. Null means nothing was rescued for that row—not that
every data-quality rule passed. This adds a column, not extra source rows.

### Why `row_difference` does fail Bronze validation

Bronze does not intentionally discard records. Its loaded counts must
match the approved snapshot counts, so `row_difference = 0` is required.

These counts are hardcoded, not dynamically read from the source. For an
approved new snapshot or stress test, update both `expected_rows` and the
count subtracted in `row_difference` for each affected table. Do not change
the rule to `>= 0`, which would also accept unintended extra rows.

## Part 0: Setup

Creates the Bronze, Silver, and Gold schemas if they do not exist. This
only prepares the Silver and Gold locations; it does not populate them.

The query compares the CSV filenames under the following volume path
with the six approved names. Missing files, unexpected CSVs, or a count
other than six cause `assert_true` to raise an error.

```text
/Volumes/workspace/default/ftw_b12_de/shared/week06/instacart_csv/
```

**Expected result:** 6 actual files, 0 unexpected files, and 0 missing files.

In [0]:
%sql
-- Owner: Nadine
-- Name: 01 - Setup
-- Purpose: Create the Bronze, Silver, Gold, and Analytics schemas and validate the six approved Instacart source CSV files.
-- Grain: One validation summary row for the Instacart source folder.

CREATE SCHEMA IF NOT EXISTS workspace.instacart_bronze;
CREATE SCHEMA IF NOT EXISTS workspace.instacart_silver;
CREATE SCHEMA IF NOT EXISTS workspace.instacart_gold;
CREATE SCHEMA IF NOT EXISTS workspace.instacart_analytics;

WITH expected_files AS (
  SELECT explode(
    array(
      'aisles.csv',
      'departments.csv',
      'order_products__prior.csv',
      'order_products__train.csv',
      'orders.csv',
      'products.csv'
    )
  ) AS file_name
),
actual_files AS (
  SELECT
    regexp_extract(path, '([^/]+)$', 1) AS file_name
  FROM read_files(
    '/Volumes/workspace/default/ftw_b12_de/shared/week06/instacart_csv/*.csv',
    format => 'binaryFile'
  )
),
file_checks AS (
  SELECT
    (SELECT COUNT(*) FROM actual_files) AS actual_file_count,
    (
      SELECT COUNT(*)
      FROM actual_files a
      LEFT ANTI JOIN expected_files e
        ON a.file_name = e.file_name
    ) AS unexpected_file_count,
    (
      SELECT COUNT(*)
      FROM expected_files e
      LEFT ANTI JOIN actual_files a
        ON e.file_name = a.file_name
    ) AS missing_file_count
)
SELECT
  actual_file_count,
  unexpected_file_count,
  missing_file_count,
  assert_true(
    actual_file_count = 6,
    'source folder must contain exactly six csv files'
  ) AS file_count_check,
  assert_true(
    unexpected_file_count = 0,
    'source folder contains an unexpected csv file'
  ) AS unexpected_file_check,
  assert_true(
    missing_file_count = 0,
    'one or more required source files are missing'
  ) AS missing_file_check
FROM file_checks;

## Part 1: Build

Loads the six source files separately to preserve their origin.
`header => true` treats the first row as column names; explicit schemas
declare the intended data types. All outputs use Delta format.

These are ingestion operations, not Silver cleaning. The only
product-specific adjustment is how quoted CSV text is parsed.

### bronze_aisles

Loads aisle IDs and names into `workspace.instacart_bronze.aisles`,
retaining `_rescued_data`.

**Grain:** One row per aisle, identified by `aisle_id`.  
**Depends on:** `setup`; no other Bronze table.  
**Expected rows:** 134.


In [0]:
%sql
-- Owner: Nadine
-- Name: 02 - Bronze Aisles
-- Purpose: Load Instacart aisle records into the Bronze Delta table using an explicit schema while retaining rescued data.
-- Grain: One row per aisle, uniquely identified by aisle_id.

USE CATALOG workspace;
CREATE SCHEMA IF NOT EXISTS instacart_bronze;
USE SCHEMA instacart_bronze;

CREATE OR REPLACE TABLE aisles
USING DELTA -- create this table as a Delta Lake table to add features on top of ordinary files, such as ACID transactions
COMMENT 'bronze copy of the instacart aisles csv' -- adds a description to the table as metadata.
AS
SELECT
  aisle_id,
  aisle,
  _rescued_data -- to capture source data that does not fit the schema specified
FROM read_files(
  '/Volumes/workspace/default/ftw_b12_de/shared/week06/instacart_csv/aisles.csv',
  format => 'csv',
  header => true,
  schema => 'aisle_id INT, aisle STRING',
  rescuedDataColumn => '_rescued_data'
);

### bronze_departments

Loads department IDs and names into
`workspace.instacart_bronze.departments`. It uses the same explicit-schema
and replacement pattern as aisles, without standardizing the source text.

**Grain:** One row per department, identified by `department_id`.  
**Depends on:** `setup`; no other Bronze table.  
**Expected rows:** 21.

In [0]:
%sql
-- Owner: Nadine
-- Name: 03 - Bronze Departments
-- Purpose: Load Instacart department records into the Bronze Delta table using an explicit schema while retaining rescued data.
-- Grain: One row per department, uniquely identified by department_id.

USE CATALOG workspace;
USE SCHEMA instacart_bronze;

CREATE OR REPLACE TABLE departments 
USING DELTA
COMMENT 'bronze copy of the instacart departments csv'
AS
SELECT
  department_id,
  department,
  _rescued_data
FROM read_files(
  '/Volumes/workspace/default/ftw_b12_de/shared/week06/instacart_csv/departments.csv',
  format => 'csv',
  header => true,
  schema => 'department_id INT, department STRING',
  rescuedDataColumn => '_rescued_data'
);

### bronze_products

Loads product IDs, names, aisle references, and department references.
The reader uses a double quote for both `quote` and `escape` to handle
embedded quotation marks and commas. This addresses the earlier parsing
issue without manually editing product names or filtering records.

**Grain:** One row per product, identified by `product_id`.  
**Depends on:** `setup`; it does not join aisles or departments here.  
**Expected rows:** 49,688.

In [0]:
%sql
-- Owner: Nadine
-- Name: 04 - Bronze Products
-- Purpose: Load Instacart product records into the Bronze Delta table using an explicit schema and correct quotation-mark parsing while retaining rescued data.
-- Grain: One row per product, uniquely identified by product_id.

USE CATALOG workspace;
USE SCHEMA instacart_bronze;

CREATE OR REPLACE TABLE workspace.instacart_bronze.products
USING DELTA
COMMENT 'bronze copy of the instacart products csv'
AS
SELECT
  product_id,
  product_name,
  aisle_id,
  department_id,
  _rescued_data
FROM read_files(
  '/Volumes/workspace/default/ftw_b12_de/shared/week06/instacart_csv/products.csv',
  format => 'csv',
  header => true,
  quote => '"',
  escape => '"',
  schema => 'product_id INT, product_name STRING, aisle_id INT, department_id INT',
  rescuedDataColumn => '_rescued_data'
);

### bronze_orders

Loads order identifiers, customer identifiers, `eval_set`, order sequence,
day/hour codes, and days since the previous order. The source's
`days_since_prior_order` nulls are preserved; validation checks that they
occur only where expected for first orders.

**Grain:** One row per order, identified by `order_id`.  
**Depends on:** `setup`; no other Bronze table.  
**Expected rows:** 3,421,083.

In [0]:
%sql
-- Owner: Nadine
-- Name: 05 - Bronze Orders
-- Purpose: Load Instacart order records into the Bronze Delta table using an explicit schema while retaining rescued data.
-- Grain: One row per Instacart order, uniquely identified by order_id.

USE CATALOG workspace;
USE SCHEMA instacart_bronze;

CREATE OR REPLACE TABLE orders 
USING DELTA
COMMENT 'bronze copy of the instacart orders csv'
AS
SELECT
  order_id,
  user_id,
  eval_set,
  order_number,
  order_dow,
  order_hour_of_day,
  days_since_prior_order,
  _rescued_data
FROM read_files(
  '/Volumes/workspace/default/ftw_b12_de/shared/week06/instacart_csv/orders.csv',
  format => 'csv',
  header => true,
  schema => 'order_id INT, user_id INT, eval_set STRING, order_number INT, order_dow INT, order_hour_of_day INT, days_since_prior_order DOUBLE',
  rescuedDataColumn => '_rescued_data'
);

### bronze_order_products_prior

Loads the prior-order product lines, including basket position and the
integer `reordered` flag. No quantity is created:
`add_to_cart_order` is position in the basket, not units purchased.

**Grain:** One product line in one prior order, identified by
`(order_id, add_to_cart_order)`.  
**Depends on:** `setup`; no other Bronze table.  
**Expected rows:** 32,434,489.

In [0]:
%sql
-- Owner: Nadine
-- Name: 06 - Bronze Order Products Prior
-- Purpose: Load prior order-product CSV records into the Bronze Delta table using an explicit schema while retaining rescued data.
-- Grain: One row per product line in one prior order, uniquely identified by (order_id, add_to_cart_order).

USE CATALOG workspace;
USE SCHEMA instacart_bronze;

CREATE OR REPLACE TABLE order_products_prior 
USING DELTA
COMMENT 'bronze copy of the instacart prior order products csv'
AS
SELECT
  order_id,
  product_id,
  add_to_cart_order,
  reordered,
  _rescued_data
FROM read_files(
  '/Volumes/workspace/default/ftw_b12_de/shared/week06/instacart_csv/order_products__prior.csv',
  format => 'csv',
  header => true,
  schema => 'order_id INT, product_id INT, add_to_cart_order INT, reordered INT',
  rescuedDataColumn => '_rescued_data'
);

### bronze_order_products_train

Loads the train-order product lines with the same declared schema as
the prior table. The two inputs remain separate in Bronze so their
source is clear; their union belongs in Silver.

**Grain:** One product line in one train order, identified by
`(order_id, add_to_cart_order)`.  
**Depends on:** `setup`; no other Bronze table.  
**Expected rows:** 1,384,617.

In [0]:
%sql
-- Owner: Nadine
-- Name: 07 - Bronze Order Products Train
-- Purpose: Load train order-product records into the Bronze Delta table using an explicit schema while retaining rescued data.
-- Grain: One row per product line in one train order, uniquely identified by (order_id, add_to_cart_order).

USE CATALOG workspace;
USE SCHEMA instacart_bronze;

CREATE OR REPLACE TABLE order_products_train 
USING DELTA
COMMENT 'bronze copy of the instacart train order products csv'
AS
SELECT
  order_id,
  product_id,
  add_to_cart_order,
  reordered,
  _rescued_data
FROM read_files(
  '/Volumes/workspace/default/ftw_b12_de/shared/week06/instacart_csv/order_products__train.csv',
  format => 'csv',
  header => true,
  schema => 'order_id INT, product_id INT, add_to_cart_order INT, reordered INT',
  rescuedDataColumn => '_rescued_data'
);

## Part 2: Validate

One consolidated query checks all 6 Bronze tables. It is read-only:
expressions such as `TRIM(aisle) = ''` inspect values without modifying them.

| Check | What it verifies |
|---|---|
| Row counts | Actual rows equal the approved snapshot counts |
| Required identifiers | Required IDs and product-line key fields are not null |
| Candidate keys | Entity IDs are unique; orders also check `(user_id, order_number)` |
| Product-line keys | Both `(order_id, add_to_cart_order)` and `(order_id, product_id)` are unique |
| Required fields | Descriptive names and required order context are present |
| Domain rules | Valid evaluation labels; day codes 0–6; hours 0–23; positive order numbers and basket positions; `reordered` is 0 or 1 |
| Prior-order interval | Nonnegative when present, null for first orders, and present for later orders |
| Rescued rows | No nonnull `_rescued_data` remains |

A table is classified as `PASS` only when its row difference and every
issue count equal zero; otherwise its status is `FAIL`.
The final windowed `assert_true` raises an error if any table fails.

**Expected result:** Six `PASS` rows. A null
`bronze_validation_check` is normal when the assertion succeeds.

**When a check fails:** The assertion can prevent the complete result
table from displaying. Inspect the same metrics with the assertion
omitted in a diagnostic copy, then keep the assertion in the pipeline
version. This is not the Silver notebook's report-only `PASS/REVIEW`
pattern.

**Counting note:** For single-column keys, the current
`COUNT(*) - COUNT(DISTINCT key)` expression also counts null-key rows.
Read it alongside `null_required_ids`; it is not a count of distinct
duplicate-key groups.

Cross-table relationship checks are left to Silver rather than repeated
as joins in this Bronze query.

In [0]:
%sql
-- Owner: Nadine
-- Name: 08 - Bronze Validation
-- Purpose: Validate Bronze row counts, required identifiers, candidate keys, required fields, domain rules, and rescued data.
-- Grain: One validation summary row per Bronze table.

WITH validation AS (

  SELECT
    'aisles' AS table_name,
    COUNT(*) AS actual_rows,
    COUNT_IF(aisle_id IS NULL) AS null_required_ids,
    COUNT(aisle_id) - COUNT(DISTINCT aisle_id)
      AS duplicate_primary_keys,
    CAST(0 AS BIGINT) AS duplicate_alternate_keys,
    COUNT_IF(
      aisle IS NULL
      OR TRIM(aisle) = ''
    ) AS required_field_issues,
    CAST(0 AS BIGINT) AS domain_issues,
    COUNT_IF(_rescued_data IS NOT NULL) AS rescued_rows
  FROM workspace.instacart_bronze.aisles

  UNION ALL

  SELECT
    'departments',
    COUNT(*),
    COUNT_IF(department_id IS NULL),
    COUNT(department_id) - COUNT(DISTINCT department_id),
    CAST(0 AS BIGINT),
    COUNT_IF(
      department IS NULL
      OR TRIM(department) = ''
    ),
    CAST(0 AS BIGINT),
    COUNT_IF(_rescued_data IS NOT NULL)
  FROM workspace.instacart_bronze.departments

  UNION ALL

  SELECT
    'products',
    COUNT(*),
    COUNT_IF(
      product_id IS NULL
      OR aisle_id IS NULL
      OR department_id IS NULL
    ),
    COUNT(product_id) - COUNT(DISTINCT product_id),
    CAST(0 AS BIGINT),
    COUNT_IF(
      product_name IS NULL
      OR TRIM(product_name) = ''
    ),
    CAST(0 AS BIGINT),
    COUNT_IF(_rescued_data IS NOT NULL)
  FROM workspace.instacart_bronze.products

  UNION ALL

  SELECT
    'orders',
    COUNT(*),
    COUNT_IF(
      order_id IS NULL
      OR user_id IS NULL
    ),
    COUNT(order_id) - COUNT(DISTINCT order_id),
    COUNT_IF(
      user_id IS NOT NULL
      AND order_number IS NOT NULL
    ) - COUNT(
      DISTINCT CASE
        WHEN user_id IS NOT NULL
          AND order_number IS NOT NULL
        THEN STRUCT(user_id, order_number)
      END
    ),
    COUNT_IF(
      eval_set IS NULL
      OR TRIM(eval_set) = ''
      OR order_number IS NULL
      OR order_dow IS NULL
      OR order_hour_of_day IS NULL
    ),
    COUNT_IF(
      eval_set NOT IN ('prior', 'train', 'test')
      OR order_number < 1
      OR order_dow NOT BETWEEN 0 AND 6
      OR order_hour_of_day NOT BETWEEN 0 AND 23
      OR days_since_prior_order < 0
      OR (
        order_number = 1
        AND days_since_prior_order IS NOT NULL
      )
      OR (
        order_number > 1
        AND days_since_prior_order IS NULL
      )
    ),
    COUNT_IF(_rescued_data IS NOT NULL)
  FROM workspace.instacart_bronze.orders

  UNION ALL

  SELECT
    'order_products_prior',
    COUNT(*),
    COUNT_IF(
      order_id IS NULL
      OR product_id IS NULL
      OR add_to_cart_order IS NULL
    ),
    COUNT_IF(
      order_id IS NOT NULL
      AND add_to_cart_order IS NOT NULL
    ) - COUNT(
      DISTINCT CASE
        WHEN order_id IS NOT NULL
          AND add_to_cart_order IS NOT NULL
        THEN STRUCT(order_id, add_to_cart_order)
      END
    ),
    COUNT_IF(
      order_id IS NOT NULL
      AND product_id IS NOT NULL
    ) - COUNT(
      DISTINCT CASE
        WHEN order_id IS NOT NULL
          AND product_id IS NOT NULL
        THEN STRUCT(order_id, product_id)
      END
    ),
    CAST(0 AS BIGINT),
    COUNT_IF(
      add_to_cart_order < 1
      OR reordered IS NULL
      OR reordered NOT IN (0, 1)
    ),
    COUNT_IF(_rescued_data IS NOT NULL)
  FROM workspace.instacart_bronze.order_products_prior

  UNION ALL

  SELECT
    'order_products_train',
    COUNT(*),
    COUNT_IF(
      order_id IS NULL
      OR product_id IS NULL
      OR add_to_cart_order IS NULL
    ),
    COUNT_IF(
      order_id IS NOT NULL
      AND add_to_cart_order IS NOT NULL
    ) - COUNT(
      DISTINCT CASE
        WHEN order_id IS NOT NULL
          AND add_to_cart_order IS NOT NULL
        THEN STRUCT(order_id, add_to_cart_order)
      END
    ),
    COUNT_IF(
      order_id IS NOT NULL
      AND product_id IS NOT NULL
    ) - COUNT(
      DISTINCT CASE
        WHEN order_id IS NOT NULL
          AND product_id IS NOT NULL
        THEN STRUCT(order_id, product_id)
      END
    ),
    CAST(0 AS BIGINT),
    COUNT_IF(
      add_to_cart_order < 1
      OR reordered IS NULL
      OR reordered NOT IN (0, 1)
    ),
    COUNT_IF(_rescued_data IS NOT NULL)
  FROM workspace.instacart_bronze.order_products_train

),

results AS (

  SELECT
    table_name,
    actual_rows,
    null_required_ids,
    duplicate_primary_keys,
    duplicate_alternate_keys,
    required_field_issues,
    domain_issues,
    rescued_rows,
    CASE
      WHEN actual_rows > 0
        AND null_required_ids = 0
        AND duplicate_primary_keys = 0
        AND duplicate_alternate_keys = 0
        AND required_field_issues = 0
        AND domain_issues = 0
        AND rescued_rows = 0
      THEN 'PASS'
      ELSE 'FAIL'
    END AS status
  FROM validation

)

SELECT
  table_name,
  actual_rows,
  null_required_ids,
  duplicate_primary_keys,
  duplicate_alternate_keys,
  required_field_issues,
  domain_issues,
  rescued_rows,
  status,
  assert_true(
    SUM(
      CASE
        WHEN status = 'FAIL' THEN 1
        ELSE 0
      END
    ) OVER () = 0,
    'one or more bronze tables failed validation; review the table-level metrics'
  ) AS bronze_validation_check
FROM results
ORDER BY table_name;

## Bronze Layer: Summary

| Table | Validation target | Notes |
|---|---|---|
| `aisles` | PASS | Source aisle lookup; no text cleaning |
| `departments` | PASS | Source department lookup; no text cleaning |
| `products` | PASS | Explicit quote/escape settings |
| `orders` | PASS | Keeps evaluation labels and first-order null intervals |
| `order_products_prior` | PASS | Prior source kept separately |
| `order_products_train` | PASS | Train source kept separately |

- All six loads use full replacement, not append.
- Every table retains `_rescued_data` for ingestion checks.
- Row differences affect Bronze status; no rows are intentionally dropped.
- Statuses above are targets, not evidence of a successful execution.

**Expected result:** All six tables pass validation before Silver starts.

**Next stage:** Silver — clean and integrate the six Bronze sources into
five logical entities and validate their relationships.

*Documentation was added to a copy of the supplied notebook. Its eight
SQL code cells were preserved unchanged, and no SQL was executed.*
